In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [13]:
# 1. Load and Prepare Dataset
print("Loading California Housing Dataset...")
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

Loading California Housing Dataset...


In [15]:
# 2. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
# 3. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [19]:
# 4. Detect Overfitting (Baseline Decision Tree)
print("\n--- Overfitting Detection ---")
tree = DecisionTreeRegressor(random_state=42) # Baseline: No constraints
tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

train_rmse = mean_squared_error(y_train, train_pred, squared=False)
test_rmse = mean_squared_error(y_test, test_pred, squared=False)

print(f"Training RMSE: {train_rmse:.4f}")
print(f"Testing RMSE: {test_rmse:.4f}")
print("Note: Large gap between Train and Test RMSE indicates Overfitting.")


--- Overfitting Detection ---
Training RMSE: 0.0000
Testing RMSE: 0.7030
Note: Large gap between Train and Test RMSE indicates Overfitting.


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [21]:
# 5. Cross-Validation (Reliable Evaluation)
print("\n--- Cross-Validation ---")
cv_scores = cross_val_score(
    tree, X_scaled, y, 
    scoring="neg_root_mean_squared_error", 
    cv=5
)
cv_rmse = -cv_scores.mean()
print(f"5-Fold CV RMSE Mean: {cv_rmse:.4f}")


--- Cross-Validation ---
5-Fold CV RMSE Mean: 0.8957


In [23]:
# 6. Hyperparameter Tuning Using GridSearchCV
print("\n--- Hyperparameter Tuning ---")
param_grid = {
    "max_depth": [3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10, 20]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42), 
    param_grid, 
    scoring="neg_root_mean_squared_error", 
    cv=5
)
grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")
best_tree = grid.best_estimator_


--- Hyperparameter Tuning ---
Best Parameters: {'max_depth': 10, 'min_samples_split': 20}


In [25]:
# 7. Evaluate Optimized Model
y_pred_tuned = best_tree.predict(X_test)
tuned_rmse = mean_squared_error(y_test, y_pred_tuned, squared=False)
tuned_r2 = r2_score(y_test, y_pred_tuned)

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
